# ANOMALYZE v2 — version Binder

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ibrahim-traore1607/anomalyze-ml/main?urlpath=lab/tree/ANOMALYZE_v2_binder.ipynb)

Analyse exploratoire et détection d'anomalies : **PCA**, **KMeans** (avec choix automatique de k) et **Isolation Forest**.

Cette version remplace l'interface Tkinter (qui nécessite un écran) par des cellules classiques, utilisables dans Binder.
Les algorithmes sont **identiques** à ceux de la v2. Seuls les fichiers CSV sont pris en charge.

**Utilisation :** exécutez les cellules dans l'ordre (*Run → Run All Cells*). Par défaut, un jeu de données de démonstration est généré automatiquement.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy  as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster       import KMeans
from sklearn.ensemble      import IsolationForest
from sklearn.metrics       import silhouette_score

%matplotlib inline

## 1. Algorithmes ML

In [ ]:
# -- best_k -------------------------------------------------------------------
def best_k(X_scaled, k_range=range(2, 7)):
    scores = {}
    for k in k_range:
        try:
            lbl = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
            scores[k] = silhouette_score(X_scaled, lbl)
        except Exception:
            pass
    return max(scores, key=scores.get) if scores else 3


# -- analyser_data ------------------------------------------------------------
def analyser_data(X_df, k=3, contamination=0.05, auto_k=False):
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_df)
    if auto_k:
        k = best_k(X_scaled)
    pca     = PCA(n_components=2)
    coords  = pca.fit_transform(X_scaled)
    var_exp = pca.explained_variance_ratio_.sum()
    labels  = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    try:
        sil = silhouette_score(X_scaled, labels)
    except Exception:
        sil = 0.0
    anom   = IsolationForest(contamination=contamination, random_state=42).fit_predict(X_scaled)
    n_anom = (anom == -1).sum()
    return {
        "coords": coords, "labels": labels, "anomalies": anom,
        "k": k, "n_anomalies": n_anom,
        "variance_explained": var_exp, "silhouette_score": sil,
        "X_df": X_df,
    }


# -- creer_visualisations (vue ensemble, conserve de v1) ----------------------
def creer_visualisations(resultats, titre="Analyse ML"):
    coords  = resultats["coords"]; labels = resultats["labels"]
    anom    = resultats["anomalies"]; X_df = resultats["X_df"]
    k       = resultats["k"]; pal = plt.cm.tab10.colors
    fig = Figure(figsize=(17, 5))
    ax1 = fig.add_subplot(131)
    for i in range(k):
        m = labels == i
        ax1.scatter(coords[m,0], coords[m,1], color=pal[i%10],
                    label=f"Cluster {i}", s=25, alpha=0.75)
    ax1.set_title(f"KMeans (k={k})", fontweight="bold")
    ax1.set_xlabel("PC1"); ax1.set_ylabel("PC2")
    ax1.legend(fontsize=8); ax1.grid(True, alpha=0.25)
    ax2 = fig.add_subplot(132)
    corr = X_df.corr()
    im   = ax2.imshow(corr, cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
    ax2.set_xticks(range(len(corr.columns))); ax2.set_yticks(range(len(corr.columns)))
    ax2.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
    ax2.set_yticklabels(corr.columns, fontsize=8)
    if len(corr.columns) <= 8:
        for r in range(len(corr.columns)):
            for c in range(len(corr.columns)):
                ax2.text(c, r, f"{corr.iloc[r,c]:.2f}", ha="center", va="center",
                         color="black", fontsize=7)
    ax2.set_title("Correlation", fontweight="bold"); fig.colorbar(im, ax=ax2)
    ax3 = fig.add_subplot(133)
    ax3.scatter(coords[anom==1,0], coords[anom==1,1], label="Normal", s=18, alpha=0.45, c="steelblue")
    ax3.scatter(coords[anom==-1,0], coords[anom==-1,1],
                label=f"Anomalie ({(anom==-1).sum()})", s=55, marker="X",
                color="red", edgecolors="black", linewidths=0.8)
    ax3.set_title("Isolation Forest", fontweight="bold")
    ax3.set_xlabel("PC1"); ax3.set_ylabel("PC2")
    ax3.legend(fontsize=8); ax3.grid(True, alpha=0.25)
    fig.suptitle(titre, fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    return fig


# -- creer_visualisations_v2 (4 figures individuelles pour les onglets) --------
def creer_visualisations_v2(resultats, titre="Analyse ML"):
    coords = resultats["coords"]; labels = resultats["labels"]
    anom   = resultats["anomalies"]; X_df = resultats["X_df"]
    k      = resultats["k"]; pal = plt.cm.tab10.colors

    rc = {
        "figure.facecolor": "white", "axes.facecolor": "#fafbfc",
        "axes.edgecolor": "#d0d7de", "axes.labelcolor": "#24292f",
        "xtick.color": "#57606a", "ytick.color": "#57606a",
        "text.color": "#24292f", "grid.color": "#e8eaed",
        "axes.titlecolor": "#24292f",
        "axes.spines.top": False, "axes.spines.right": False,
    }

    def draw_cluster(ax):
        for i in range(k):
            m = labels == i
            ax.scatter(coords[m,0], coords[m,1], color=pal[i%10],
                       label=f"Cluster {i}  (n={m.sum()})", s=32, alpha=0.78,
                       edgecolors="white", linewidths=0.4)
        ax.set_title(f"KMeans Clustering  (k={k})", fontweight="bold", fontsize=11)
        ax.set_xlabel("Composante 1  (PC1)", fontsize=9)
        ax.set_ylabel("Composante 2  (PC2)", fontsize=9)
        ax.legend(fontsize=8, framealpha=0.8); ax.grid(True, alpha=0.35, linestyle="--")

    def draw_corr(fig_r, ax):
        corr = X_df.corr()
        im   = ax.imshow(corr, cmap="coolwarm", aspect="auto",
                         vmin=-1, vmax=1, interpolation="nearest")
        ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=9)
        ax.set_yticklabels(corr.columns, fontsize=9)
        if len(corr.columns) <= 10:
            for r in range(len(corr.columns)):
                for c in range(len(corr.columns)):
                    v = corr.iloc[r, c]
                    ax.text(c, r, f"{v:.2f}", ha="center", va="center",
                            color="white" if abs(v)>0.6 else "#24292f", fontsize=8)
        ax.set_title("Matrice de Correlation (Pearson)", fontweight="bold", fontsize=11)
        fig_r.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    def draw_anom(ax):
        nn = (anom==1).sum(); na = (anom==-1).sum()
        ax.scatter(coords[anom==1,0], coords[anom==1,1],
                   label=f"Normal  (n={nn})", s=22, alpha=0.45, c="#0969da")
        ax.scatter(coords[anom==-1,0], coords[anom==-1,1],
                   label=f"Anomalie  (n={na})", s=65, marker="X",
                   color="#cf222e", edgecolors="#24292f", linewidths=0.8)
        ax.set_title("Isolation Forest  - Detection d'Anomalies",
                     fontweight="bold", fontsize=11)
        ax.set_xlabel("Composante 1  (PC1)", fontsize=9)
        ax.set_ylabel("Composante 2  (PC2)", fontsize=9)
        ax.legend(fontsize=8, framealpha=0.8); ax.grid(True, alpha=0.35, linestyle="--")

    with plt.rc_context(rc):
        # Vue d'ensemble
        fig_all = Figure(figsize=(17, 5))
        ax1 = fig_all.add_subplot(131); ax2 = fig_all.add_subplot(132)
        ax3 = fig_all.add_subplot(133)
        draw_cluster(ax1); draw_corr(fig_all, ax2); draw_anom(ax3)
        fig_all.suptitle(titre, fontsize=13, fontweight="bold", y=1.01)
        fig_all.tight_layout()

        # Clustering seul
        fig_c = Figure(figsize=(9, 6))
        draw_cluster(fig_c.add_subplot(111))
        fig_c.suptitle(f"Clustering KMeans  -  {titre}", fontsize=12, fontweight="bold")
        fig_c.tight_layout()

        # Correlation seule
        fig_cr = Figure(figsize=(9, 7))
        draw_corr(fig_cr, fig_cr.add_subplot(111))
        fig_cr.suptitle(f"Matrice de Correlation  -  {titre}", fontsize=12, fontweight="bold")
        fig_cr.tight_layout()

        # Anomalies seules
        fig_a = Figure(figsize=(9, 6))
        draw_anom(fig_a.add_subplot(111))
        fig_a.suptitle(f"Anomalies  -  {titre}", fontsize=12, fontweight="bold")
        fig_a.tight_layout()

    return {"all": fig_all, "cluster": fig_c, "corr": fig_cr, "anom": fig_a}

In [ ]:
FICHIER = None   # ex. "mes_donnees.csv"

def charger_dataset(filepath):
    """Charge un CSV et ne garde que les colonnes numériques."""
    try:
        df = pd.read_csv(filepath)
    except Exception:
        df = pd.read_csv(filepath, encoding="latin1")
    X = df.select_dtypes(include=[np.number])
    return X, None, os.path.basename(filepath)


def donnees_demo(seed=42):
    """3 groupes de points + quelques valeurs aberrantes."""
    rng = np.random.default_rng(seed)
    centres = [(0, 0, 0, 0, 0), (6, 6, 2, -3, 4), (-5, 5, -4, 3, -2)]
    blocs = [rng.normal(c, 1.0, size=(300, 5)) for c in centres]
    aberrants = rng.uniform(-14, 14, size=(15, 5))
    X = np.vstack(blocs + [aberrants])
    cols = ["mesure_1", "mesure_2", "mesure_3", "mesure_4", "mesure_5"]
    return pd.DataFrame(X, columns=cols), None, "Données de démonstration"


if FICHIER is None:
    X, col_couleur, titre = donnees_demo()
else:
    X, col_couleur, titre = charger_dataset(FICHIER)

print(f"{titre} : {X.shape[0]} lignes, {X.shape[1]} colonnes")
X.head()

## 3. Paramètres de l'analyse

- `COLONNES` : colonnes à analyser (`None` = toutes les colonnes numériques)
- `K` : nombre de clusters
- `AUTO_K` : si `True`, k est choisi automatiquement (score de silhouette)
- `CONTAMINATION` : proportion d'anomalies attendue (entre 0 et 0.5)

In [ ]:
COLONNES      = None
K             = 3
AUTO_K        = False
CONTAMINATION = 0.05

X_df = (X if COLONNES is None else X[COLONNES]).dropna()
assert X_df.shape[1] >= 2, "Sélectionnez au moins 2 colonnes."
assert len(X_df) >= 10,    "Moins de 10 observations."

resultats = analyser_data(X_df, k=K, contamination=CONTAMINATION, auto_k=AUTO_K)

print(f"Observations analysées     : {len(X_df)}")
print(f"Nombre de clusters (k)     : {resultats['k']}")
print(f"Anomalies détectées        : {resultats['n_anomalies']}")
print(f"Variance expliquée (PCA 2D): {resultats['variance_explained']:.1%}")
print(f"Score de silhouette        : {resultats['silhouette_score']:.3f}")

## 4. Visualisations

In [ ]:
figs = creer_visualisations_v2(resultats, titre)
display(figs["all"])

In [ ]:
display(figs["cluster"])
display(figs["corr"])
display(figs["anom"])

## 5. Anomalies détectées

In [ ]:
out = X_df.copy()
out["cluster"] = resultats["labels"]
out["anomalie"] = resultats["anomalies"] == -1

print("Taille des clusters :")
print(out["cluster"].value_counts().sort_index().to_string())
out[out["anomalie"]].head(20)

## 6. Export (facultatif)

In [ ]:
out.to_csv("resultats_anomalyze.csv", index=False)
figs["all"].savefig("anomalyze_vue_ensemble.png", dpi=150, bbox_inches="tight")
print("Fichiers créés : resultats_anomalyze.csv, anomalyze_vue_ensemble.png")